# Conversion GGUF (Bonsai 27B) -> SafeTensors Ternaire AI Box

Ce notebook convertit un modèle `.gguf` (ex. Bonsai 27B `q1_0`) en format **AI Box** (`model.safetensors` + `quant_config.json` + `config.json` + `tokenizer.json`) avec rotation **Hadamard `H_128`** et packing **2-bit** (`00 -> 0`, `01 -> +1`, `11 -> -1`), **sans aucun entraînement**.

In [ ]:
# CELLULE 1 : Cloner le dépôt ai-box & installer les dépendances
!git clone https://github.com/thienban/ai-box.git || true
%cd /content/ai-box
!git pull
!pip install -q gguf safetensors huggingface_hub torch numpy

In [ ]:
# CELLULE 2 : Télécharger le .gguf depuis Hugging Face (ou définir GGUF_PATH manuellement)
from huggingface_hub import hf_hub_download

HF_REPO_ID = "prism-ml/Bonsai-27B-gguf"  # Adaptez selon le dépôt GGUF choisi
GGUF_FILENAME = "Bonsai-27B-q1_0.gguf"   # Adaptez au nom exact du fichier .gguf

GGUF_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename=GGUF_FILENAME)
print("Fichier GGUF prêt :", GGUF_PATH)

In [ ]:
# CELLULE 3 : Conversion GGUF -> SafeTensors AI Box
# Mettre --max-layers 2 pour un test rapide (~10s), ou retirer --max-layers pour les 64 couches complètes.
!python scripts/convert_gguf_to_aibox.py \
    --gguf "{GGUF_PATH}" \
    --out /content/bonsai-aibox \
    --max-layers 2

!ls -lh /content/bonsai-aibox

In [ ]:
# CELLULE 4 (Optionnel) : Vérifier l'export avec le binaire Rust `aibox inspect`
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
!~/.cargo/bin/cargo run --release --manifest-path app/Cargo.toml -p aibox -- inspect /content/bonsai-aibox

In [ ]:
# CELLULE 5 (Optionnel) : Pousser le modèle converti vers votre Hugging Face (pour `aibox pull` sur PC)
# from huggingface_hub import HfApi
# api = HfApi(token="hf_VOTRE_TOKEN")
# api.create_repo(repo_id="votre-user/bonsai-27b-aibox", exist_ok=True, private=True)
# api.upload_folder(folder_path="/content/bonsai-aibox", repo_id="votre-user/bonsai-27b-aibox")